In [ ]:
# !git clone https://github.com/vuthetam/mscoco_image_captioning.git

# import os
# import sys
# sys.path.append("/kaggle/working/mscoco_image_captioning")

In [ ]:
# %%writefile mscoco_image_captioning/main.py

import os
import pandas as pd
from torch.utils.data import DataLoader
from accelerate import Accelerator
from accelerate.utils import set_seed
from torch.optim import AdamW
from torch import nn

from vocabulary import Vocabulary
from config import Backbone
from dataset import MSCOCODataset, create_img_transform
from model.encoder import create_encoder
from model.decoder import CaptionDecoder
from checkpoint import load_checkpoint, save_checkpoint
from engine import train_one_epoch, evaluate_one_epoch


BACKBONE: Backbone = "clip_vit_b16"
DATASET_COCO_PATH = "dataset/dataset_coco.json"
IMAGES_DIR = "dataset/images"
# DATASET_COCO_PATH = "/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json"
# IMAGES_DIR = "/kaggle/input/datasets/vuthetam/mscoco-2014/images"

SEED = 42
BATCH_SIZE = 32
MIN_FREQ = 5
NUM_EPOCHS = 10

D_MODEL = 512
NHEAD = 8
NUM_LAYERS = 4
MAX_LEN = 32
DROPOUT = 0.1

LEARNING_RATE = 1e-4

BEAM_SIZE = 5
LENGTH_PENALTY = 0.7

SAVE_CHECKPOINT_DIR = "checkpoints/" + BACKBONE
os.makedirs(SAVE_CHECKPOINT_DIR, exist_ok=True)
LOAD_CHECKPOINT_DIR = "checkpoints/" + BACKBONE
SAVE_LAST_CHECKPOINT_PATH = os.path.join(SAVE_CHECKPOINT_DIR, "last_checkpoint.pt")
SAVE_BEST_CHECKPOINT_PATH = os.path.join(SAVE_CHECKPOINT_DIR, "best_checkpoint.pt")
LOAD_LAST_CHECKPOINT_PATH = os.path.join(LOAD_CHECKPOINT_DIR, "last_checkpoint.pt")
LOAD_BEST_CHECKPOINT_PATH = os.path.join(LOAD_CHECKPOINT_DIR, "best_checkpoint.pt")

In [ ]:
# %%writefile -a mscoco_image_captioning/main.py

df = pd.read_json(DATASET_COCO_PATH)
images_df = pd.json_normalize(df["images"], record_path="sentences", meta=["filepath", "filename", "split"])
train_df = images_df[images_df["split"].isin(["train", "restval"])]
val_df = images_df[images_df["split"].isin(["val"])]

tokens_list = train_df["tokens"].tolist()
vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build_from_tokens(tokens_list)

img_transform = create_img_transform(BACKBONE)
train_dataset = MSCOCODataset(IMAGES_DIR, train_df, img_transform, vocab, MAX_LEN)
val_dataset = MSCOCODataset(IMAGES_DIR, val_df, img_transform, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, BATCH_SIZE, num_workers=4)



In [ ]:
# %%writefile -a mscoco_image_captioning/main.py

def train(train_loader, val_loader, vocab):
    accelerator = Accelerator(mixed_precision="fp16")
    set_seed(SEED)

    vocab_size = len(vocab)
    accelerator.print(f"Vocab size: {vocab_size}")

    encoder = create_encoder(BACKBONE, D_MODEL)
    decoder = CaptionDecoder(vocab_size, D_MODEL, MAX_LEN, NHEAD, DROPOUT, NUM_LAYERS)

    criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_token_id)

    params = list(filter(
        lambda p: p.requires_grad,
        list(encoder.parameters()) + list(decoder.parameters())
    ))
    optimizer = AdamW(params, LEARNING_RATE)

    start_epoch = 0
    best_val_loss = float("inf")
    if os.path.isfile(LOAD_LAST_CHECKPOINT_PATH):
        start_epoch, _, best_val_loss = load_checkpoint(LOAD_LAST_CHECKPOINT_PATH, encoder, decoder, accelerator.device, optimizer)
        accelerator.print("Load checkpoint successfully")

    encoder, decoder, optimizer, train_loader, val_loader = accelerator.prepare(
        encoder, decoder, optimizer, train_loader, val_loader
    )

    accelerator.print(f"Start epoch: {start_epoch} | best val loss: {best_val_loss}")

    for epoch_idx in range(start_epoch, NUM_EPOCHS):
        train_loss = train_one_epoch(encoder, decoder, train_loader, optimizer, criterion, accelerator, show_progress=True)
        val_loss = evaluate_one_epoch(encoder, decoder, val_loader, criterion, accelerator)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(SAVE_BEST_CHECKPOINT_PATH, encoder, decoder, optimizer, epoch_idx+1, train_loss, best_val_loss, accelerator)

        save_checkpoint(SAVE_LAST_CHECKPOINT_PATH, encoder, decoder, optimizer, epoch_idx+1, train_loss, best_val_loss, accelerator)

        accelerator.print(f"Epoch {epoch_idx+1} / {NUM_EPOCHS} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

    accelerator.wait_for_everyone()
    accelerator.end_training()

train(train_loader, val_loader, vocab)

In [ ]:
# !accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 mscoco_image_captioning/main.py